In [0]:
# nb_read_control_table
# Parameters passed by ADF via Databricks job parameters:
#   source_name  (string) — which row to read

dbutils.widgets.text("source_name", "payments")
source_name = dbutils.widgets.get("source_name")

row = spark.sql(f"""
    SELECT endpoint, sink_base_path, file_prefix, page_size, load_type, last_watermark
    FROM ev_catalog.ctl.pipeline_control
    WHERE source_name = '{source_name}'
      AND is_active = true
""").collect()

if not row:
    raise Exception(f"No active control row for source_name='{source_name}'")

r = row[0]

# Return values to ADF via notebook exit value (JSON string)
import json
result = {
    "endpoint":       r["endpoint"],
    "sink_base_path": r["sink_base_path"],
    "file_prefix":    r["file_prefix"],
    "page_size":      int(r["page_size"]),
    "load_type":      r["load_type"],
    "last_watermark": r["last_watermark"]
}

dbutils.notebook.exit(json.dumps(result))